### Loading the model

In [3]:
import torch
from pathlib import Path

from models_v2 import (
    ImprovedGenerator,
    LargePatchGANDiscriminator
)

# --------------------------------------------------
# Select device
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# --------------------------------------------------
# Recreate the same model architecture
# --------------------------------------------------

G_AB = ImprovedGenerator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

G_BA = ImprovedGenerator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

D_A = LargePatchGANDiscriminator(
    input_channels=3
).to(device)

D_B = LargePatchGANDiscriminator(
    input_channels=3
).to(device)


# --------------------------------------------------
# Checkpoint path
# --------------------------------------------------

checkpoint_dir = Path(
    r"/app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/checkpoints_v2"
)

best_checkpoint_candidates = sorted(
    checkpoint_dir.glob("best*.pth")
)

print("Best checkpoint candidates:")

for path in best_checkpoint_candidates:
    print(path)

if len(best_checkpoint_candidates) == 0:
    raise FileNotFoundError(
        "No checkpoint beginning with 'best' was found."
    )

best_checkpoint_path = best_checkpoint_candidates[0]


# --------------------------------------------------
# Load trained weights
# --------------------------------------------------

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device
)

G_AB.load_state_dict(checkpoint["G_AB"])
G_BA.load_state_dict(checkpoint["G_BA"])
D_A.load_state_dict(checkpoint["D_A"])
D_B.load_state_dict(checkpoint["D_B"])


# --------------------------------------------------
# Set evaluation mode
# --------------------------------------------------

G_AB.eval()
G_BA.eval()
D_A.eval()
D_B.eval()

print("\nLoaded checkpoint successfully.")
print("Checkpoint:", best_checkpoint_path)
print("Saved epoch:", checkpoint.get("epoch"))
print("Monitor score:", checkpoint.get("monitor_score"))

Device: cuda
GPU: NVIDIA GeForce RTX 4090
Best checkpoint candidates:
/app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/checkpoints_v2/best_cyclegan_v2.pth

Loaded checkpoint successfully.
Checkpoint: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/checkpoints_v2/best_cyclegan_v2.pth
Saved epoch: 100
Monitor score: 0.2727239729243923


### Create the Inference Folders

In [4]:
from pathlib import Path

# Use the dataset path visible to your new notebook
DATASET_ROOT = Path(
    "/app/DATA266-Lab1-Pair45/task3_GAN/dataset"
)

MONET_SOURCE = DATASET_ROOT / "monet_jpg"
PHOTO_SOURCE = DATASET_ROOT / "photo_jpg"

print("Monet folder:", MONET_SOURCE)
print("Photo folder:", PHOTO_SOURCE)

print("Monet folder exists:", MONET_SOURCE.exists())
print("Photo folder exists:", PHOTO_SOURCE.exists())

if MONET_SOURCE.exists():
    monet_files = [
        p for p in MONET_SOURCE.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    print("Monet images:", len(monet_files))

if PHOTO_SOURCE.exists():
    photo_files = [
        p for p in PHOTO_SOURCE.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    print("Photo images:", len(photo_files))

Monet folder: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/monet_jpg
Photo folder: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/photo_jpg
Monet folder exists: True
Photo folder exists: True
Monet images: 300
Photo images: 7038


### Predict Monet → Photo images

In [5]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
import torch


# --------------------------------------------------
# Inference dataset
# --------------------------------------------------

class InferenceImageDataset(Dataset):

    def __init__(self, folder):
        self.paths = sorted([
            path for path in Path(folder).iterdir()
            if path.suffix.lower() in {
                ".jpg", ".jpeg", ".png"
            }
        ])

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5)
            )
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image_path = self.paths[index]

        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        return image


# --------------------------------------------------
# Save generated tensor as JPG
# --------------------------------------------------

def save_generated_image(image_tensor, output_path):
    image_tensor = image_tensor.detach().cpu()

    # Convert from [-1, 1] to [0, 1]
    image_tensor = (image_tensor + 1.0) / 2.0
    image_tensor = image_tensor.clamp(0, 1)

    image = to_pil_image(image_tensor)
    image.save(output_path, format="JPEG", quality=95)


# --------------------------------------------------
# Create Monet DataLoader
# --------------------------------------------------

monet_dataset = InferenceImageDataset(MONET_SOURCE)

monet_loader = DataLoader(
    monet_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Monet images available:", len(monet_dataset))


# --------------------------------------------------
# Output folder
# --------------------------------------------------

OUTPUT_ROOT = Path.cwd() / "kaggle_finetuning_outputs_v2"
MONET_TO_PHOTO_DIR = OUTPUT_ROOT / "pred_A2B"

MONET_TO_PHOTO_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# Generate Monet → Photo images
# --------------------------------------------------

G_BA.eval()

generated_count = 0

with torch.no_grad():

    for batch_index, monet_images in enumerate(monet_loader, start=1):

        monet_images = monet_images.to(device)

        # G_BA performs Monet → Photo
        generated_photos = G_BA(monet_images)

        for generated_photo in generated_photos:

            if generated_count >= 300:
                break

            output_path = (
                MONET_TO_PHOTO_DIR
                / f"monet_to_photo_{generated_count:05d}.jpg"
            )

            save_generated_image(
                generated_photo,
                output_path
            )

            generated_count += 1

        print(
            f"Processed batch [{batch_index}/{len(monet_loader)}] | "
            f"Images saved: {generated_count}",
            flush=True
        )

        if generated_count >= 300:
            break


print("\nMonet → Photo generation completed.")
print("Images generated:", generated_count)
print("Saved in:", MONET_TO_PHOTO_DIR.resolve())

Monet images available: 300
Processed batch [1/19] | Images saved: 16
Processed batch [2/19] | Images saved: 32
Processed batch [3/19] | Images saved: 48
Processed batch [4/19] | Images saved: 64
Processed batch [5/19] | Images saved: 80
Processed batch [6/19] | Images saved: 96
Processed batch [7/19] | Images saved: 112
Processed batch [8/19] | Images saved: 128
Processed batch [9/19] | Images saved: 144
Processed batch [10/19] | Images saved: 160
Processed batch [11/19] | Images saved: 176
Processed batch [12/19] | Images saved: 192
Processed batch [13/19] | Images saved: 208
Processed batch [14/19] | Images saved: 224
Processed batch [15/19] | Images saved: 240
Processed batch [16/19] | Images saved: 256
Processed batch [17/19] | Images saved: 272
Processed batch [18/19] | Images saved: 288
Processed batch [19/19] | Images saved: 300

Monet → Photo generation completed.
Images generated: 300
Saved in: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/kaggle_finetuning_outputs_v2/pred

In [ ]:
C:\Sam\DATA266-Lab1-Pair45\task3_GAN\Samruddhi\kaggle_outputs_v2

### Predict Photo → Monet images

In [6]:
# --------------------------------------------------
# Generate Photo → Monet images
# --------------------------------------------------

photo_dataset = InferenceImageDataset(PHOTO_SOURCE)

photo_loader = DataLoader(
    photo_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)


PHOTO_TO_MONET_DIR = (
    OUTPUT_ROOT / "pred_B2A"
)

PHOTO_TO_MONET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

G_AB.eval()

generated_count = 0

with torch.no_grad():

    for batch_index, photo_images in enumerate(
        photo_loader,
        start=1
    ):

        photo_images = photo_images.to(device)

        # G_AB performs Photo → Monet
        generated_monets = G_AB(photo_images)

        for generated_monet in generated_monets:

            if generated_count >= 300:
                break

            output_path = (
                PHOTO_TO_MONET_DIR
                / f"photo_to_monet_{generated_count:05d}.jpg"
            )

            save_generated_image(
                generated_monet,
                output_path
            )

            generated_count += 1

        print(
            f"Processed batch [{batch_index}/{len(photo_loader)}] | "
            f"Images saved: {generated_count}",
            flush=True
        )

        if generated_count >= 300:
            break


print("\nPhoto → Monet generation completed.")
print("Images generated:", generated_count)
print("Saved in:", PHOTO_TO_MONET_DIR.resolve())

Processed batch [1/440] | Images saved: 16
Processed batch [2/440] | Images saved: 32
Processed batch [3/440] | Images saved: 48
Processed batch [4/440] | Images saved: 64
Processed batch [5/440] | Images saved: 80
Processed batch [6/440] | Images saved: 96
Processed batch [7/440] | Images saved: 112
Processed batch [8/440] | Images saved: 128
Processed batch [9/440] | Images saved: 144
Processed batch [10/440] | Images saved: 160
Processed batch [11/440] | Images saved: 176
Processed batch [12/440] | Images saved: 192
Processed batch [13/440] | Images saved: 208
Processed batch [14/440] | Images saved: 224
Processed batch [15/440] | Images saved: 240
Processed batch [16/440] | Images saved: 256
Processed batch [17/440] | Images saved: 272
Processed batch [18/440] | Images saved: 288
Processed batch [19/440] | Images saved: 300

Photo → Monet generation completed.
Images generated: 300
Saved in: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/kaggle_finetuning_outputs_v2/pred_B2A
